In [ ]:
!cp /content/drive/MyDrive/School/University-of-Guelph/MDS/Course/26-2S_DATA6700/1_data/data_siamese_gradcam.zip /content/

In [ ]:
!unzip -q /content/data_siamese_gradcam.zip -d /content/data

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from torch.utils.data import DataLoader, Dataset

In [ ]:
DIR_DATA_DRIVE = Path("data")
DIR_DATA_LOCAL = Path("/content/data")

# ============================================================
# Directories
# ============================================================
DIR_DATA = DIR_DATA_LOCAL
DIR_METADATA = DIR_DATA_DRIVE / "0_metadata"

DIR_ICEYE = DIR_DATA / "1_ICEYE"
DIR_SENTINEL = DIR_DATA / "2_Sentinel2"
DIR_TERRAIN = DIR_DATA / "3_Terrain" / "depmap"

FILEPATH_PAIR_MANIFEST = DIR_METADATA / "pair-manifest.csv"

# ============================================================
# Dataset
# ============================================================
PATCH_SIZE = 256
NUM_CHANNELS = 3
NUM_CLASSES = 2
METADATA_DIM = 4
CONTROLLED = True

# ============================================================
# DataLoader
# ============================================================
BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = True
SHUFFLE_TRAIN = True
SHUFFLE_VALID = False

# ============================================================
# Training
# ============================================================
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = "cuda"

# ============================================================
# Misc
# ============================================================
SEED = 42


In [ ]:
# if not DIR_DATA_LOCAL.exists():
#     print("Copying dataset to local disk...")
#     shutil.copytree(DIR_DATA_DRIVE, DIR_DATA_LOCAL)
#     print("Done!")
# else:
#     print("Dataset already exists.")

In [ ]:
class MetadataNormalizer:
    """
    Normalize continuous metadata and encode categorical metadata.
    """
    def __init__(self):
        self.delta_mean = None
        self.delta_std = None
        self.angle_mean = None
        self.angle_std = None

    def fit(self, pair_manifest):
        """
        Compute normalization statistics from the training manifest.
        """
        train = pair_manifest[pair_manifest["split"] == "train"]

        self.delta_mean = train["delta_days"].mean()
        self.delta_std = train["delta_days"].std()

        self.angle_mean = train["incidence_angle"].mean()
        self.angle_std = train["incidence_angle"].std()

        print("Metadata statistics")
        print(
            f"delta_days : "
            f"{self.delta_mean:.2f} ± {self.delta_std:.2f}"
        )
        print(
            f"incidence_angle : "
            f"{self.angle_mean:.2f} ± {self.angle_std:.2f}"
        )

    def transform(self, sample):
        """
        Convert one sample into a normalized metadata vector.
        """
        delta_days = (
            sample["delta_days"] - self.delta_mean
        ) / self.delta_std

        incidence_angle = (
            sample["incidence_angle"] - self.angle_mean
        ) / self.angle_std

        orbit_direction = (
            1.0
            if sample["orbit_direction"] == "Ascending"
            else 0.0
        )

        look_side = (
            1.0
            if sample["look_side"] == "Right"
            else 0.0
        )

        return np.array(
            [
                delta_days,
                incidence_angle,
                orbit_direction,
                look_side,
            ],
            dtype=np.float32,
        )

class FloodDataset(Dataset):
    def __init__(
        self,
        split: str,
        metadata_normalizer=None,
        api_decay=0.8,
        controlled=False,
    ):
        """
        split:
            "train", "valid", or "test"
        """
        self.metadata_normalizer = metadata_normalizer
        self.samples = []
        self.target_suffix = "_controlled" if controlled else ""

        pair_manifest = pd.read_csv(
            DIR_DATA / FILEPATH_PAIR_MANIFEST
        )

        # --------------------------------------------------
        # Filter pairs
        # --------------------------------------------------
        if split not in ("train", "valid", "test"):
            raise ValueError("split must be 'train', 'valid', or 'test'")

        pair_manifest = pair_manifest[pair_manifest["split"] == split]

        # --------------------------------------------------
        # Build sample list
        # --------------------------------------------------
        for _, pair in pair_manifest.iterrows():

            sar_base_dir = DIR_ICEYE / pair["sar_base"]
            sar_target_dir = DIR_ICEYE / pair["sar_target"]

            # Rainfall representation
            rain_history = np.array(
                [
                    pair["precip_-1"],
                    pair["precip_-2"],
                    pair["precip_-3"],
                    pair["precip_-4"],
                    pair["precip_-5"],
                ],
                dtype=np.float32,
            )
            weights = np.array(
                [api_decay ** i for i in range(5)],
                dtype=np.float32,
            )
            api = np.dot(weights, rain_history).astype(np.float32)

            # Every SAR patch becomes one sample
            for sar_path in sorted(sar_base_dir.glob("*.tif")):

                parts = sar_path.stem.split("_", 1)
                if len(parts) != 2:
                    print(f"Unexpected filename: {sar_path.name}")
                    continue

                patch_id = parts[1]

                # Check split
                split_in_patch_id = get_split_from_patch_filename(
                    patch_id
                )
                if split not in split_in_patch_id:
                    continue

                year = get_year_from_patch_id(patch_id)

                base_iceye_id = pair["sar_base"].split("_")[-1]
                target_iceye_id = pair["sar_target"].split("_")[-1]

                sar_target_path = (
                    sar_target_dir
                    / f"{target_iceye_id}_{patch_id}{self.target_suffix}.tif"
                )

                if not sar_target_path.exists():
                    sar_target_path = (
                        sar_target_dir
                        / f"{target_iceye_id}_{patch_id}.tif"
                    )

                ndvi_target_path = (
                    DIR_SENTINEL /
                    f"patches_{year}" /
                    pair["ndvi_target"] /
                    f"{pair['ndvi_target']}_{patch_id}_NDVI{self.target_suffix}.tif"
                )

                if not ndvi_target_path.exists():
                    ndvi_target_path = (
                        DIR_SENTINEL /
                        f"patches_{year}" /
                        pair["ndvi_target"] /
                        f"{pair['ndvi_target']}_{patch_id}_NDVI.tif"
                    )

                sample = {
                    "pair_id": pair["pair_id"],
                    "patch_id": patch_id,

                    "label":
                        1 if pair["pair_id"].startswith("W")
                        else 0,

                    "sar_base":
                        sar_base_dir /
                        f"{base_iceye_id}_{patch_id}.tif",

                    "sar_target":
                        sar_target_path,

                    "ndvi_base":
                        DIR_SENTINEL /
                        f"patches_{year}" /
                        pair["ndvi_base"] /
                        f"{pair['ndvi_base']}_{patch_id}_NDVI.tif",

                    "ndvi_target":
                        ndvi_target_path,

                    "terrain":
                        DIR_TERRAIN /
                        f"patches_{year}" /
                        f"depmap_{patch_id}.tif",

                    # --------------------------
                    # Raw metadata
                    # --------------------------
                    "delta_days":
                        pair["delta_days"],

                    "incidence_angle":
                        pair["incidence_angle"],

                    "orbit_direction":
                        pair["orbit_direction"],

                    "look_side":
                        pair["look_side"],

                    # --------------------------
                    # Rainfall target
                    # --------------------------
                    "rain":
                        api,
                }

                # Skip incomplete samples
                paths = [
                    sample["sar_base"],
                    sample["sar_target"],
                    sample["ndvi_base"],
                    sample["ndvi_target"],
                    sample["terrain"],
                ]

                if all(p.exists() for p in paths):
                    self.samples.append(sample)

        print(f"{split}: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def _read_tiff(self, path):
        with rasterio.open(path) as src:
            img = src.read(1).astype(np.float32)
        return img

    def __getitem__(self, idx):

        sample = self.samples[idx]

        # ----------------------------
        # Load images
        # ----------------------------
        sar_base = self._read_tiff(sample["sar_base"])
        sar_target = self._read_tiff(sample["sar_target"])
        ndvi_base = self._read_tiff(sample["ndvi_base"])
        ndvi_target = self._read_tiff(sample["ndvi_target"])
        terrain = self._read_tiff(sample["terrain"])

        # ----------------------------
        # Build image tensors
        # ----------------------------
        before = np.stack([
            sar_base,
            ndvi_base,
            terrain,
        ])
        after = np.stack([
            sar_target,
            ndvi_target,
            terrain,
        ])
        before = torch.from_numpy(before)
        after = torch.from_numpy(after)

        # ----------------------------
        # Label
        # ----------------------------
        label = torch.tensor(sample["label"], dtype=torch.float32)

        # ----------------------------
        # Metadata
        # ----------------------------
        metadata = self.metadata_normalizer.transform(sample)
        metadata = torch.from_numpy(metadata)

        # ----------------------------
        # Rain target
        # ----------------------------
        rain = torch.tensor(sample["rain"], dtype=torch.float32)

        return {
            "before": before,
            "after": after,
            "label": label,
            "rain": rain,
            "metadata": metadata,
            "pair_id": sample["pair_id"],
            "patch_id": sample["patch_id"],
        }


def get_year_from_patch_id(patch_id: str) -> int:
    """
    Extract the year from a patch ID.

    Examples
    --------
    B24BC0120 -> 2024
    W25CB0120 -> 2025
    """

    return 2000 + int(patch_id[2:4])

from pathlib import Path

def get_split_from_patch_filename(patch_id: str) -> str:
    """
    Extract the dataset split from a SAR patch ID.

    Examples
    --------
    TB24CC0120 -> "train/valid"
    EB24CC0120 -> "test"
    """
    split = patch_id[:1]
    if split == "T":
        return "train/valid"
    elif split == "E":
        return "test"

    raise ValueError(
        f"Unexpected split '{split}' in {patch_id}"
    )


In [ ]:
pair_manifest = pd.read_csv(FILEPATH_PAIR_MANIFEST)

metadata_normalizer = MetadataNormalizer()
metadata_normalizer.fit(pair_manifest)

train_dataset = FloodDataset(
    "train",
    metadata_normalizer=metadata_normalizer,
)

valid_dataset = FloodDataset(
    "valid",
    metadata_normalizer=metadata_normalizer,
)

test_dataset = FloodDataset(
    "test",
    metadata_normalizer=metadata_normalizer,
    controlled=CONTROLLED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE_TRAIN,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE_VALID,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

In [ ]:
sample = test_dataset[13]

print(f"pair_id  : {sample['pair_id']}")
print(f"patch_id : {sample['patch_id']}")
print(f"label    : {sample['label']}")
print(f"metadata : {sample['metadata']}")
print(f"rain     : {sample['rain']}")

print(sample["before"].shape)
print(sample["after"].shape)

CHANNEL_NAMES = [
    "SAR",
    "NDVI",
    "Terrain"
]
fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8)
)
for i in range(3):
    axes[0, i].imshow(
        sample["before"][i],
        cmap="gray"
    )
    axes[0, i].set_title(
        f"Before ({CHANNEL_NAMES[i]})"
    )
    axes[0, i].axis("off")
    axes[1, i].imshow(
        sample["after"][i],
        cmap="gray"
    )
    axes[1, i].set_title(
        f"After ({CHANNEL_NAMES[i]})"
    )
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
batch = next(iter(train_loader))

print(batch["before"].shape)
print(batch["after"].shape)
print(batch["label"].shape)

In [ ]:
print(torch.isnan(batch["before"]).any())
print(torch.isnan(batch["after"]).any())

In [ ]:
import copy
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

In [ ]:
DIR_CHECKPOINTS = Path("/content/drive/MyDrive/School/University-of-Guelph/MDS/Course/26-2S_DATA6700/2_model_checkpoints")
DIR_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)
CHECKPOINT_NAME_BASE = "siamese_gradcam_reg"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device: {DEVICE}")

EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

In [ ]:
class MultiScalePooling(nn.Module):
    """
    Multi-scale global pooling using
    Adaptive Average Pooling + Adaptive Max Pooling.
    """

    def __init__(self):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):

        avg = self.avg_pool(x)
        mx = self.max_pool(x)

        x = torch.cat([avg, mx], dim=1)

        return x.flatten(1)


class SiameseFloodNetRegression(nn.Module):
    def __init__(
        self,
        metadata_dim=4,
        pretrained=True,
    ):
        super().__init__()

        # --------------------------------------------------
        # Shared ResNet18 encoder
        # --------------------------------------------------
        backbone = models.resnet18(
            weights=(
                models.ResNet18_Weights.DEFAULT
                if pretrained
                else None
            )
        )

        # Replace first convolution for custom input
        backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )

        # Remove average pooling and FC layer
        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        # --------------------------------------------------
        # Image pooling
        # --------------------------------------------------
        self.pool = MultiScalePooling()

        # Before (1024) + After (1024)
        image_feature_dim = 512 * 2 * 2

        # --------------------------------------------------
        # Metadata encoder
        # --------------------------------------------------
        self.metadata_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 16),
            nn.ReLU(inplace=True),

            nn.Linear(16, 32),
            nn.ReLU(inplace=True),
        )

        # --------------------------------------------------
        # Regression head
        # --------------------------------------------------
        self.regression_head = nn.Sequential(
            nn.Linear(image_feature_dim + 32, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(128, 1),
        )

    def forward(
        self,
        before,
        after,
        metadata,
    ):
        # --------------------------------------------------
        # Shared encoder
        # --------------------------------------------------
        feat_before = self.encoder(before)
        feat_after = self.encoder(after)

        # --------------------------------------------------
        # Multi-scale pooling
        # --------------------------------------------------
        feat_before = self.pool(feat_before)
        feat_after = self.pool(feat_after)

        # --------------------------------------------------
        # Concatenate image features
        # --------------------------------------------------
        image_feature = torch.cat(
            [
                feat_before,
                feat_after,
            ],
            dim=1,
        )

        # --------------------------------------------------
        # Metadata branch
        # --------------------------------------------------
        metadata_feature = self.metadata_encoder(
            metadata
        )

        # --------------------------------------------------
        # Late fusion
        # --------------------------------------------------
        feature = torch.cat(
            [
                image_feature,
                metadata_feature,
            ],
            dim=1,
        )

        # --------------------------------------------------
        # Regression
        # --------------------------------------------------
        output = self.regression_head(feature)

        return output.squeeze(1)

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):
    model.train()

    running_loss = 0.0
    running_mae = 0.0
    total = 0

    for batch in loader:

        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)
        rain = batch["rain"].to(DEVICE)

        optimizer.zero_grad()

        output = model(
            before,
            after,
            metadata
        )

        loss = criterion(output, rain)

        loss.backward()
        optimizer.step()

        batch_size = rain.size(0)

        running_loss += loss.item() * batch_size

        with torch.no_grad():
            mae = torch.mean(torch.abs(output - rain))

        running_mae += mae.item() * batch_size

        total += batch_size

    epoch_loss = running_loss / total
    epoch_mae = running_mae / total

    return epoch_loss, epoch_mae


@torch.no_grad()
def validate(
    model,
    loader,
    criterion
):
    model.eval()

    running_loss = 0.0
    running_mae = 0.0
    total = 0

    for batch in loader:

        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)
        rain = batch["rain"].to(DEVICE)

        output = model(
            before,
            after,
            metadata
        )

        loss = criterion(output, rain)

        batch_size = rain.size(0)

        running_loss += loss.item() * batch_size

        mae = torch.mean(torch.abs(output - rain))
        running_mae += mae.item() * batch_size

        total += batch_size

    epoch_loss = running_loss / total
    epoch_mae = running_mae / total

    return epoch_loss, epoch_mae

In [ ]:
# Initialize model
model = SiameseFloodNetRegression(metadata_dim=METADATA_DIM).to(DEVICE)
print(model)

In [ ]:
# Test model
batch = next(iter(train_loader))

before = batch["before"].to(DEVICE)
after = batch["after"].to(DEVICE)
metadata = batch["metadata"].to(DEVICE)
rain = batch["rain"].to(DEVICE)

output = model(
    before,
    after,
    metadata
)
print(output.shape)
# torch.Size([32])

In [ ]:
criterion = nn.SmoothL1Loss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [ ]:
best_valid_loss = float("inf")
history = []

start = time.time()

for epoch in range(EPOCHS):
    # train_loss, train_acc = train_one_epoch(
    train_loss, train_mae = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )
    # valid_loss, valid_acc = validate(
    valid_loss, valid_mae = validate(
        model,
        valid_loader,
        criterion
    )

    scheduler.step()

    # Save epoch results
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_mae": train_mae,
        "valid_loss": valid_loss,
        "valid_mae": valid_mae,
    })

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train MAE {train_mae:.3f} | "
        f"Valid Loss {valid_loss:.4f} | "
        f"Valid MAE {valid_mae:.3f}"
    )

    # if valid_acc > best_accuracy:
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = f"{CHECKPOINT_NAME_BASE}_{timestamp}_epoch{str(epoch + 1).zfill(2)}_loss{valid_loss:.4f}.pth"
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "valid_loss": valid_loss,
                "valid_mae": valid_mae,
            },
            DIR_CHECKPOINTS / model_name
        )
        print("✓ Best model updated.")

elapsed = time.time() - start

# Convert to DataFrame
log_train_df = pd.DataFrame(history)

print(f"\nTraining finished in {elapsed/60:.1f} minutes.")
print(f"Best validation loss: {best_valid_loss:.4f}")

In [ ]:
### SiameseFloodNetRegression ###

log = """
Epoch 001 | Train Loss 12.0727 | Train MAE 12.558 | Valid Loss 14.5038 | Valid MAE 15.004
✓ Best model updated.
Epoch 002 | Train Loss 11.0434 | Train MAE 11.526 | Valid Loss 15.6248 | Valid MAE 16.110
Epoch 003 | Train Loss 10.2000 | Train MAE 10.656 | Valid Loss 14.3888 | Valid MAE 14.880
✓ Best model updated.
Epoch 004 | Train Loss 8.8332 | Train MAE 9.281 | Valid Loss 12.6673 | Valid MAE 13.158
✓ Best model updated.
Epoch 005 | Train Loss 7.2046 | Train MAE 7.654 | Valid Loss 11.8047 | Valid MAE 12.261
✓ Best model updated.
Epoch 006 | Train Loss 6.5255 | Train MAE 6.955 | Valid Loss 11.8179 | Valid MAE 12.277
Epoch 007 | Train Loss 5.6276 | Train MAE 6.038 | Valid Loss 11.1361 | Valid MAE 11.629
✓ Best model updated.
Epoch 008 | Train Loss 4.8612 | Train MAE 5.265 | Valid Loss 9.5132 | Valid MAE 9.974
✓ Best model updated.
Epoch 009 | Train Loss 4.5274 | Train MAE 4.919 | Valid Loss 10.0325 | Valid MAE 10.481
Epoch 010 | Train Loss 4.1721 | Train MAE 4.554 | Valid Loss 11.5880 | Valid MAE 11.972
Epoch 011 | Train Loss 3.3390 | Train MAE 3.717 | Valid Loss 9.1621 | Valid MAE 9.628
✓ Best model updated.
Epoch 012 | Train Loss 3.0945 | Train MAE 3.463 | Valid Loss 10.5526 | Valid MAE 10.936
Epoch 013 | Train Loss 3.0019 | Train MAE 3.366 | Valid Loss 9.5143 | Valid MAE 9.981
Epoch 014 | Train Loss 2.9976 | Train MAE 3.358 | Valid Loss 8.8799 | Valid MAE 9.342
✓ Best model updated.
Epoch 015 | Train Loss 2.8341 | Train MAE 3.180 | Valid Loss 9.1227 | Valid MAE 9.540
Epoch 016 | Train Loss 2.5020 | Train MAE 2.856 | Valid Loss 9.4285 | Valid MAE 9.837
Epoch 017 | Train Loss 2.5587 | Train MAE 2.919 | Valid Loss 8.4754 | Valid MAE 8.893
✓ Best model updated.
Epoch 018 | Train Loss 2.2768 | Train MAE 2.625 | Valid Loss 9.2668 | Valid MAE 9.664
Epoch 019 | Train Loss 2.2943 | Train MAE 2.643 | Valid Loss 8.8138 | Valid MAE 9.204
Epoch 020 | Train Loss 1.8899 | Train MAE 2.226 | Valid Loss 9.0355 | Valid MAE 9.465
Epoch 021 | Train Loss 2.1030 | Train MAE 2.446 | Valid Loss 9.5011 | Valid MAE 9.883
Epoch 022 | Train Loss 1.9083 | Train MAE 2.244 | Valid Loss 8.5852 | Valid MAE 9.005
Epoch 023 | Train Loss 1.9667 | Train MAE 2.305 | Valid Loss 9.2464 | Valid MAE 9.650
Epoch 024 | Train Loss 1.7979 | Train MAE 2.136 | Valid Loss 9.0384 | Valid MAE 9.448
Epoch 025 | Train Loss 1.8634 | Train MAE 2.197 | Valid Loss 9.2754 | Valid MAE 9.677
Epoch 026 | Train Loss 1.8956 | Train MAE 2.238 | Valid Loss 9.3417 | Valid MAE 9.764
Epoch 027 | Train Loss 1.9276 | Train MAE 2.277 | Valid Loss 8.8650 | Valid MAE 9.277
Epoch 028 | Train Loss 1.3652 | Train MAE 1.695 | Valid Loss 9.0820 | Valid MAE 9.510
Epoch 029 | Train Loss 1.5475 | Train MAE 1.882 | Valid Loss 9.2659 | Valid MAE 9.674
Epoch 030 | Train Loss 1.5984 | Train MAE 1.923 | Valid Loss 9.1933 | Valid MAE 9.608
Epoch 031 | Train Loss 1.4800 | Train MAE 1.804 | Valid Loss 9.0913 | Valid MAE 9.506
Epoch 032 | Train Loss 1.4461 | Train MAE 1.769 | Valid Loss 9.4669 | Valid MAE 9.887
Epoch 033 | Train Loss 1.6916 | Train MAE 2.021 | Valid Loss 9.6564 | Valid MAE 10.063
Epoch 034 | Train Loss 1.5116 | Train MAE 1.835 | Valid Loss 9.2787 | Valid MAE 9.686
Epoch 035 | Train Loss 1.4815 | Train MAE 1.806 | Valid Loss 9.1653 | Valid MAE 9.580
Epoch 036 | Train Loss 1.3566 | Train MAE 1.684 | Valid Loss 9.1397 | Valid MAE 9.562
Epoch 037 | Train Loss 1.2885 | Train MAE 1.610 | Valid Loss 9.1102 | Valid MAE 9.528
Epoch 038 | Train Loss 1.3146 | Train MAE 1.627 | Valid Loss 9.2220 | Valid MAE 9.629
Epoch 039 | Train Loss 1.3751 | Train MAE 1.698 | Valid Loss 9.1846 | Valid MAE 9.597
Epoch 040 | Train Loss 1.3401 | Train MAE 1.662 | Valid Loss 9.2569 | Valid MAE 9.654
Epoch 041 | Train Loss 1.3565 | Train MAE 1.674 | Valid Loss 9.3303 | Valid MAE 9.731
Epoch 042 | Train Loss 1.3544 | Train MAE 1.671 | Valid Loss 9.0461 | Valid MAE 9.456
Epoch 043 | Train Loss 1.2155 | Train MAE 1.528 | Valid Loss 9.3133 | Valid MAE 9.720
Epoch 044 | Train Loss 1.1404 | Train MAE 1.457 | Valid Loss 9.1880 | Valid MAE 9.609
Epoch 045 | Train Loss 1.3185 | Train MAE 1.639 | Valid Loss 9.1741 | Valid MAE 9.596
Epoch 046 | Train Loss 1.1618 | Train MAE 1.477 | Valid Loss 9.0453 | Valid MAE 9.461
Epoch 047 | Train Loss 1.1487 | Train MAE 1.461 | Valid Loss 9.2612 | Valid MAE 9.680
Epoch 048 | Train Loss 1.3760 | Train MAE 1.699 | Valid Loss 9.2058 | Valid MAE 9.622
Epoch 049 | Train Loss 1.2607 | Train MAE 1.577 | Valid Loss 9.2799 | Valid MAE 9.693
Epoch 050 | Train Loss 1.1495 | Train MAE 1.469 | Valid Loss 9.1997 | Valid MAE 9.615
"""

# Training finished in 12.5 minutes.
# Best validation loss: 8.4754


In [ ]:
import re

import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel

class EpochMetrics(BaseModel):
    epoch: int
    train_loss: float
    train_mae: float
    valid_loss: float
    valid_mae: float

pattern = re.compile(
    r"Epoch\s+(\d+)"
    r"\s+\|\s+Train Loss\s+([\d.]+)"
    r"\s+\|\s+Train MAE\s+([\d.]+)"
    r"\s+\|\s+Valid Loss\s+([\d.]+)"
    r"\s+\|\s+Valid MAE\s+([\d.]+)"
)

history: list[EpochMetrics] = []

for match in pattern.finditer(log):
    epoch, train_loss, train_mae, valid_loss, valid_mae = match.groups()

    history.append(
        EpochMetrics(
            epoch=epoch,
            train_loss=train_loss,
            train_mae=train_mae,
            valid_loss=valid_loss,
            valid_mae=valid_mae,
        )
    )

log_train_df = pd.DataFrame(
    x.model_dump()
    for x in history
)

In [ ]:
best_idx_loss = log_train_df["valid_loss"].idxmin()
best_epoch_loss = log_train_df.loc[best_idx_loss, "epoch"]
best_loss = log_train_df.loc[best_idx_loss, "valid_loss"]

plt.figure(figsize=(8, 5))

plt.plot(
    log_train_df["epoch"],
    log_train_df["train_loss"],
    label="Training"
)

plt.plot(
    log_train_df["epoch"],
    log_train_df["valid_loss"],
    label="Validation"
)

plt.scatter(
    best_epoch_loss,
    best_loss,
    color="red",
    zorder=3,
    label=f"Best validation ({best_epoch_loss})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss (Regression)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
best_idx_mae = log_train_df["valid_mae"].idxmin()
best_epoch_mae = log_train_df.loc[best_idx_mae, "epoch"]
best_mae = log_train_df.loc[best_idx_mae, "valid_mae"]

plt.figure(figsize=(8, 5))

plt.plot(
    log_train_df["epoch"],
    log_train_df["train_mae"],
    label="Training"
)

plt.plot(
    log_train_df["epoch"],
    log_train_df["valid_mae"],
    label="Validation"
)

plt.scatter(
    best_epoch_mae,
    best_mae,
    color="red",
    zorder=3,
    label=f"Best validation ({best_epoch_mae})"
)

plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("Training and Validation MAE (Regression)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
model = SiameseFloodNetRegression(metadata_dim=METADATA_DIM).to(DEVICE)
checkpoint = torch.load(
    DIR_CHECKPOINTS / "siamese_gradcam_reg_20260721_234903_epoch17_loss8.4754.pth",
    map_location=DEVICE
)
model.load_state_dict(
    checkpoint["model_state_dict"]
)
model.eval()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [ ]:
@torch.no_grad()
def evaluate_test(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    for batch in loader:
        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)
        rain = batch["rain"].to(DEVICE)

        prediction = model(before, after, metadata)

        y_true.append(rain.cpu().numpy())
        y_pred.append(prediction.cpu().numpy())

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"MAE : {mae:.3f} mm")
    print(f"RMSE: {rmse:.3f} mm")
    print(f"R²  : {r2:.3f}")

    return y_true, y_pred

In [ ]:
y_true, y_pred = evaluate_test(
    model,
    test_loader
)

In [ ]:
# MAE : 13.257 mm
# RMSE: 15.698 mm
# R²  : -0.157

### CONTROLLED ###
# MAE : 13.964 mm
# RMSE: 16.743 mm
# R²  : -0.316

In [ ]:
!pip install grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np

In [ ]:
class AfterBranchWrapper(nn.Module):
    def __init__(self, model, before, metadata):
        super().__init__()
        self.model = model
        self.before = before
        self.metadata = metadata

    def forward(self, after):
        output = self.model(
            self.before,
            after,
            self.metadata
        )

        # Grad-CAM expects shape (batch_size, num_outputs)
        return output.unsqueeze(1)


def show_gradcam(
    model,
    dataset,
    index,
    device=DEVICE
):
    """
    Display Grad-CAM for one sample.

    Parameters
    ----------
    model : SiameseFloodNetLateFusion
    dataset : FloodDataset
    index : int
        Sample index.
    """

    sample = dataset[index]

    before = sample["before"].unsqueeze(0).to(device)
    after = sample["after"].unsqueeze(0).to(device)
    metadata = sample["metadata"].unsqueeze(0).to(device)

    # --------------------------------------------------
    # Prediction
    # --------------------------------------------------
    model.eval()

    with torch.no_grad():
        prediction = model(
            before,
            after,
            metadata
        ).item()

    ground_truth = sample["rain"].item()

    # --------------------------------------------------
    # Images
    # --------------------------------------------------
    before_img = before.squeeze().cpu().numpy()[0]
    after_img = after.squeeze().cpu().numpy()[0]

    before_img = (
        before_img - before_img.min()
    ) / (
        before_img.max() - before_img.min() + 1e-6
    )

    after_img = (
        after_img - after_img.min()
    ) / (
        after_img.max() - after_img.min() + 1e-6
    )

    # --------------------------------------------------
    # Grad-CAM
    # --------------------------------------------------
    wrapped_model = AfterBranchWrapper(
        model,
        before,
        metadata
    )

    cam = GradCAM(
        model=wrapped_model,
        target_layers=[model.encoder[-1]]
    )

    grayscale_cam = cam(
        input_tensor=after,
        targets=[ClassifierOutputTarget(0)]
    )[0]

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------
    fig, ax = plt.subplots(
        2,
        2,
        figsize=(10, 10),
        constrained_layout=True
    )

    ax[0, 0].imshow(before_img, cmap="gray")
    ax[0, 0].set_title("SAR Before")
    ax[0, 0].axis("off")

    ax[0, 1].imshow(after_img, cmap="gray")
    ax[0, 1].set_title("SAR After")
    ax[0, 1].axis("off")

    ax[1, 0].imshow(before_img, cmap="gray")

    im = ax[1, 0].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1
    )

    ax[1, 0].set_title("Grad-CAM")
    ax[1, 0].axis("off")

    ax[1, 1].imshow(after_img, cmap="gray")
    ax[1, 1].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1
    )

    ax[1, 1].set_title("Grad-CAM")
    ax[1, 1].axis("off")

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.7,
        label="Importance"
    )

    error = prediction - ground_truth

    fig.suptitle(
        f"Pair : {sample['pair_id']}\n"
        f"Patch: {sample['patch_id']}\n"
        f"Predicted API : {prediction:.2f}\n"
        f"Ground Truth API : {ground_truth:.2f}\n"
        f"Error : {error:+.2f}",
        fontsize=13,
    )

    plt.show()

In [ ]:
show_gradcam(
    model,
    test_dataset,
    12,
)

In [ ]:
from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class AfterBranchWrapper(nn.Module):
    def __init__(self, model, before, metadata):
        super().__init__()
        self.model = model
        self.before = before
        self.metadata = metadata

    def forward(self, after):
        output = self.model(
            self.before,
            after,
            self.metadata
        )
        return output.unsqueeze(1)


def show_layercam(
    model,
    dataset,
    index,
    target_layers=None,
    device=DEVICE,
):
    """
    Display Layer-CAM for one sample.

    Parameters
    ----------
    model : nn.Module
        Trained Siamese classification model.

    dataset : FloodDataset

    index : int
        Sample index.

    target_layers : list[nn.Module] or None
        Encoder layer(s) to visualize.
        If None, uses the last encoder layer.
    """

    sample = dataset[index]

    before = sample["before"].unsqueeze(0).to(device)
    after = sample["after"].unsqueeze(0).to(device)
    metadata = sample["metadata"].unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        logit = model(
            before,
            after,
            metadata
        )

        probability = torch.sigmoid(logit).item()
        prediction = int(probability >= 0.5)

    ground_truth = int(sample["label"].item())

    # --------------------------------------------------
    # Images
    # --------------------------------------------------

    before_img = before.squeeze().cpu().numpy()[0]
    after_img = after.squeeze().cpu().numpy()[0]

    before_img = (
        before_img - before_img.min()
    ) / (
        before_img.max() - before_img.min() + 1e-6
    )

    after_img = (
        after_img - after_img.min()
    ) / (
        after_img.max() - after_img.min() + 1e-6
    )

    # --------------------------------------------------
    # Layer selection
    # --------------------------------------------------

    if target_layers is None:
        target_layers = [model.encoder[-1]]

    elif not isinstance(target_layers, list):
        target_layers = [target_layers]

    # --------------------------------------------------
    # Layer-CAM
    # --------------------------------------------------

    wrapped_model = AfterBranchWrapper(
        model,
        before,
        metadata
    )

    cam = LayerCAM(
        model=wrapped_model,
        target_layers=target_layers
    )

    grayscale_cam = cam(
        input_tensor=after,
        targets=[ClassifierOutputTarget(0)]
    )[0]

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(
        2,
        2,
        figsize=(10, 10),
        constrained_layout=True
    )

    ax[0, 0].imshow(before_img, cmap="gray")
    ax[0, 0].set_title("SAR Before")
    ax[0, 0].axis("off")

    ax[0, 1].imshow(after_img, cmap="gray")
    ax[0, 1].set_title("SAR After")
    ax[0, 1].axis("off")

    ax[1, 0].imshow(before_img, cmap="gray")

    im = ax[1, 0].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1,
    )

    ax[1, 0].set_title("Layer-CAM")
    ax[1, 0].axis("off")

    ax[1, 1].imshow(after_img, cmap="gray")
    ax[1, 1].imshow(
        grayscale_cam,
        cmap="jet",
        alpha=0.4,
        vmin=0,
        vmax=1,
    )

    ax[1, 1].set_title("Layer-CAM")
    ax[1, 1].axis("off")

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.7,
        label="Importance",
    )

    correct = prediction == ground_truth

    layer_names = ", ".join(
        [type(layer).__name__ for layer in target_layers]
    )

    fig.suptitle(
        f"Pair : {sample['pair_id']}\n"
        f"Patch: {sample['patch_id']}\n"
        f"Predicted probability : {probability:.3f}\n"
        f"Prediction : {prediction}\n"
        f"Ground Truth : {ground_truth}\n"
        f"Layers : {layer_names}\n"
        f"{'Correct' if correct else 'Incorrect'}",
        fontsize=13,
    )

    plt.show()

In [ ]:
show_layercam(
    model,
    test_dataset,
    12,
    target_layers=[
        model.encoder[-3],
        model.encoder[-2],
        model.encoder[-1],
    ],
)

In [ ]:
from pathlib import Path
import rasterio
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

In [ ]:
if CONTROLLED:
    DIR_GRADCAM = DIR_DATA_DRIVE / "3_results" / "siamese-gradcam-regression" / "patches-gradcam-controlled"
else:
    DIR_GRADCAM = DIR_DATA_DRIVE / "3_results" / "siamese-gradcam-regression" / "patches-gradcam"
DIR_GRADCAM.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_gradcam_geotiffs(
    model,
    dataset,
    output_dir,
    device=DEVICE,
    controlled=False,
):
    """
    Export Grad-CAM GeoTIFFs for every sample.

    Output filename:
        gradcam_{pair_id}_{patch_id}_api{prediction}.tif
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    suffix = "_controlled" if controlled else ""

    model.eval()

    target_layers = [model.encoder[-1]]

    for idx in range(len(dataset)):

        tensor_sample = dataset[idx]
        file_sample = dataset.samples[idx]

        before = tensor_sample["before"].unsqueeze(0).to(device)
        after = tensor_sample["after"].unsqueeze(0).to(device)
        metadata = tensor_sample["metadata"].unsqueeze(0).to(device)

        # --------------------------------------------------
        # Prediction
        # --------------------------------------------------
        with torch.no_grad():
            prediction = model(
                before,
                after,
                metadata
            ).item()

        # --------------------------------------------------
        # Grad-CAM
        # --------------------------------------------------
        wrapped_model = AfterBranchWrapper(
            model,
            before,
            metadata
        )

        cam = GradCAM(
            model=wrapped_model,
            target_layers=target_layers
        )

        grayscale_cam = cam(
            input_tensor=after,
            targets=[ClassifierOutputTarget(0)]
        )[0]

        # --------------------------------------------------
        # Save GeoTIFF
        # --------------------------------------------------
        with rasterio.open(file_sample["sar_target"]) as src:

            profile = src.profile.copy()
            profile.update(
                driver="GTiff",
                dtype="float32",
                count=1,
                compress="LZW"
            )

            output_path = (
                output_dir /
                f"gradcam_"
                f"{tensor_sample['pair_id']}_"
                f"{tensor_sample['patch_id']}_"
                f"pred{prediction:.3f}"
                f"{suffix}.tif"
            )

            with rasterio.open(
                output_path,
                "w",
                **profile
            ) as dst:
                dst.write(
                    grayscale_cam.astype(np.float32),
                    1
                )

        print(
            f"[{idx+1:3d}/{len(dataset)}] "
            f"{output_path.name}"
        )

    print("Done!")

In [ ]:
export_gradcam_geotiffs(
    model,
    test_dataset,
    DIR_GRADCAM,
    controlled=CONTROLLED
)